In [13]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [12]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [14]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [15]:
openai= OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages
)
response.choices[0].message.content

'Hi! Welcome, and nice to meet you. I’m here to help with questions, explanations, writing, coding, planning, and more. What would you like to do today?\n\nHere are a few ideas:\n- Explain a topic in simple terms\n- Draft or edit an email, paragraph, or story\n- Help with math or coding problems\n- Plan a trip, project, or study schedule\n- Translate or summarize text\n\nTell me what you’re interested in, or share a task you have in mind and we’ll dive in.'

In [17]:
polito= fetch_website_contents("https://www.polito.it/")
print(polito[:500])

Politecnico di Torino

Salta al contenuto principale
Stato Servizi
Header
Login
IT
EN
Apri/chiudi il menu delle lingue
Salta al contenuto principale
Navigazione principale
Ateneo
Ateneo
Attiva/disattiva il sotto-menu
L'Ateneo
Vai alla pagina Ateneo
Chi siamo
Colpo d'occhio
Strategia
Un Ateneo internazionale
Parità, welfare e inclusione
Campus sostenibile
Career Hub
Lavora e collabora con noi
Qualità
Comunicazione e ufficio stampa
Sostieni il Politecnico
Didattica
Didattica
Attiva/disattiva il so


In [18]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a wicked assistant that analyzes the contents of a website,
and provides a short, humorous and wicked summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown. keep the summary to a maximum of 10 sentences.
"""

In [19]:
user_prompt_prefix = """
Here is the content of a website - please analyze it and provide a short, humorous and wicked summary.    
If it includes any new events please include them in the summary, and if it includes any information about the university's history, please ignore that - we just want to know what's new and interesting!
"""

In [20]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [21]:
messages_for(polito)


[{'role': 'system',
  'content': '\nYou are a wicked assistant that analyzes the contents of a website,\nand provides a short, humorous and wicked summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown. keep the summary to a maximum of 10 sentences.\n'},
 {'role': 'user',
  'content': "\nHere is the content of a website - please analyze it and provide a short, humorous and wicked summary.    \nIf it includes any new events please include them in the summary, and if it includes any information about the university's history, please ignore that - we just want to know what's new and interesting!\nPolitecnico di Torino\n\nSalta al contenuto principale\nStato Servizi\nHeader\nLogin\nIT\nEN\nApri/chiudi il menu delle lingue\nSalta al contenuto principale\nNavigazione principale\nAteneo\nAteneo\nAttiva/disattiva il sotto-menu\nL'Ateneo\nVai alla pagina Ateneo\nChi siamo\nColpo d'occhio\nStrategia

In [22]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [23]:
summarize("https://www.polito.it/")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [24]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [26]:
display_summary("https://www.polito.it/")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}